# Entity Reviews Analysis

This notebook analyzes review sentiment and entity relevance for the entity-driven app.
- Review-level sentiment (multilingual).
- Entity-match score for selected entities and suggested new entities per review.


In [8]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline

DB_PATH = Path('../data/app.sqlite')
if not DB_PATH.exists():
    raise FileNotFoundError(f'DB not found at {DB_PATH.resolve()}')

conn = sqlite3.connect(DB_PATH)

In [9]:
df_tables = pd.read_sql(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
      AND name NOT LIKE 'sqlite_%'
    ORDER BY name
    """,
    conn
)
df_tables

,name
0,edge_relations
1,edges
2,node_type
3,nodes
4,review
5,review_sentiment
6,user


In [10]:
df = pd.read_sql(
    """
    SELECT *
    FROM review_sentiment
    """,
    conn
)
df.tail()

,review_id,sentiment_raw,confidence,method,version,created_at
27,19,-0.923525,0.923525,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-06T09:55:53.558607+00:00
28,20,-0.782086,0.782086,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-06T09:55:53.558607+00:00
29,21,0.000000,0.466579,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-06T09:55:53.558607+00:00
30,22,-0.707653,0.707653,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-06T09:55:53.558607+00:00
31,26,-0.890821,0.890821,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-06T09:55:53.558607+00:00


In [12]:
df = pd.read_sql(
    """
    SELECT *
    FROM review
    """,
    conn
)
df.tail()

,id,user_id,content,node_id,entity_name,created_at,updated_at
38,40,2,뜨거운 아메리카노 중간 사이즈 1600원. 향긋하니 맛있었다.,98,메머드커피,2026-01-07T20:34:32.966Z,2026-01-13T06:11:28.352Z
39,41,1,광어회가 4만원인데 3만원에 세일한다기에 밤에 갔다가 추워서 사람이 없어 그런가 2...,97,이마트 광어회,2026-01-08T14:00:23.694Z,2026-01-13T06:11:15.304Z
40,42,1,이마트에서 6개 만원 세일하길래 구입. 프리미엄 몰츠 좋아하는데 이건 그냥 그랬다....,96,산토리 나마비루,2026-01-09T15:24:14.040Z,2026-01-13T06:10:33.704Z
41,43,1,저번에 이마트에서 6개 만원 세일했는데 요샌 기린 맥주와 함께 만원 5개 세일중,87,산토리 나마 비어,2026-01-09T15:25:07.414Z,2026-01-09T15:25:07.414Z
42,45,2,부대전골이라는데 전골이라기엔 건더기가 적지만 부대찌개라면 건더기 푸짐하다. 대파와 ...,95,양반 그릴리 부대전골,2026-01-11T12:08:45.301Z,2026-01-13T06:10:22.992Z


In [2]:
df = pd.read_sql(
    #SELECT relation, is_transitive, default_weight, allowed_parent_types, allowed_child_types 
    #SELECT relation, is_transitive, allowed_parent_types, allowed_child_types 
    
    """
    SELECT relation, is_transitive, default_weight, allowed_parent_types, allowed_child_types 
    FROM edge_relations
    """,
    conn
)
df.to_dict()

{'relation': {0: 'sells',
  1: 'contains',
  2: 'operates',
  3: 'produces',
  4: 'located_in',
  5: 'part_of',
  6: 'exclusive_brand_of',
  7: 'provides',
  8: 'publishes'},
 'is_transitive': {0: 0, 1: 1, 2: 0, 3: 0, 4: 1, 5: 1, 6: 0, 7: 0, 8: 0},
 'default_weight': {0: 1.0,
  1: 0.3,
  2: 0.7,
  3: 0.9,
  4: 0.4,
  5: 0.25,
  6: 0.75,
  7: 0.95,
  8: 0.85},
 'allowed_parent_types': {0: '["restaurant","vendor","venue"]',
  1: '["location","menu","product","venue"]',
  2: '["brand"]',
  3: '["brand"]',
  4: '["content_provider","restaurant","service_provider","vendor","venue"]',
  5: '["menu","product"]',
  6: '["brand"]',
  7: '["service_provider","venue"]',
  8: '["content_provider"]'},
 'allowed_child_types': {0: '["product"]',
  1: '["location","menu","product","venue"]',
  2: '["restaurant","vendor","venue"]',
  3: '["product"]',
  4: '["location","venue"]',
  5: '["menu","product"]',
  6: '["brand"]',
  7: '["service"]',
  8: '["content"]'}}

In [18]:
df

,relation,is_transitive,default_weight,allowed_parent_types,allowed_child_types
0,sells,0,1.0,"[""restaurant"",""vendor"",""venue""]","[""product""]"
1,contains,1,1.0,"[""location"",""menu"",""product"",""venue""]","[""location"",""menu"",""product"",""venue""]"
2,operates,0,1.0,"[""brand""]","[""restaurant"",""vendor"",""venue""]"
3,produces,0,1.0,"[""brand""]","[""product""]"
4,located_in,1,1.0,"[""content_provider"",""restaurant"",""service_prov...","[""location"",""venue""]"
5,part_of,1,1.0,"[""menu"",""product""]","[""menu"",""product""]"
6,exclusive_brand_of,0,1.0,"[""brand""]","[""brand""]"
7,provides,0,1.0,"[""service_provider"",""venue""]","[""service""]"
8,publishes,0,1.0,"[""content_provider""]","[""content""]"


In [16]:
df = pd.read_sql(
    """
    SELECT node_type, base_prior
    FROM node_type_prior
    """,
    conn
)
df['node_type'].to_list()

['product',
 'brand',
 'vendor',
 'menu',
 'restaurant',
 'venue',
 'location',
 'service',
 'default',
 'service_provider',
 'content_provider',
 'content']

In [4]:
df = pd.read_sql(
    """
    SELECT *
    FROM review_sentiment
    """,
    conn
)
df

,review_id,sentiment_raw,confidence,method,version,created_at
0,1,0.000000,0.000000,rule,v1,2026-01-06T09:47:10.876447+00:00
1,2,0.000000,0.000000,rule,v1,2026-01-06T09:47:10.876447+00:00
2,4,0.000000,0.000000,rule,v1,2026-01-06T09:47:10.876447+00:00
3,5,1.000000,0.045455,rule,v1,2026-01-06T09:47:10.876447+00:00
4,6,-2.000000,0.058824,rule,v1,2026-01-06T09:47:10.876447+00:00
5,7,1.000000,0.066667,rule,v1,2026-01-06T09:47:10.876447+00:00
6,11,0.000000,0.000000,rule,v1,2026-01-06T09:47:10.876447+00:00
7,14,1.000000,0.027027,rule,v1,2026-01-06T09:47:10.876447+00:00
8,15,0.000000,0.000000,rule,v1,2026-01-06T09:47:10.876447+00:00
9,17,0.000000,0.000000,rule,v1,2026-01-06T09:47:10.876447+00:00


In [20]:
df.loc[df['alias'].str.startswith('굽')]

,alias,node_id
11,굽네치킨,20
34,굽네,20


In [4]:
reviews = pd.read_sql('SELECT id, content, created_at, updated_at, user_id FROM review', conn)
nodes = pd.read_sql('SELECT id, name, type FROM nodes', conn)
review_entities = pd.read_sql('SELECT * FROM review_entity', conn)
review_entity_sentiments = pd.read_sql('SELECT * FROM review_entity_sentiment', conn)
conn.close()

print('reviews', reviews.shape)
print('nodes', nodes.shape)
print('review_entities', review_entities.shape)

reviews (42, 5)
nodes (78, 3)
review_entities (70, 4)


In [6]:
review_entities.loc[review_entities.alias == '굽네치킨']

,id,review_id,node_id,alias
11,12,9,20,굽네치킨


In [9]:
nodes.loc[nodes.id ==20]

,id,name,type
11,20,굽네치킨,brand


In [12]:
nodes.loc[nodes.name == '굽네']

,id,name,type


In [13]:
nodes

,id,name,type
0,1,쏘마치,product
1,2,K마라치킨,product
2,3,멕시카나,brand
3,6,노랑통닭,brand
4,9,배달의민족,service_provider
...,...,...,...
73,84,광어회,menu
74,85,병원,service
75,86,배달,service
76,87,산토리 나마비루,product


In [66]:
reviews['content'].map(len).max()
#nodes
#review_entities

np.int64(194)

In [67]:
df_re = review_entities.set_index('id')
df_r = reviews.set_index('id')

In [68]:
review_entity_sentiments.sort_values('sentiment_raw', ascending=True)

,review_entity_id,sentiment_raw,confidence,method,version,created_at
6,7,-2.000000,0.058824,rule,v1,2026-01-06T09:47:10.876447+00:00
33,34,-1.000000,0.055556,rule,v1,2026-01-06T09:47:10.876447+00:00
74,26,-0.923525,0.923525,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-06T09:55:53.558607+00:00
82,34,-0.890821,0.890821,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-06T09:55:53.558607+00:00
69,21,-0.851396,0.851396,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-06T09:55:53.558607+00:00
...,...,...,...,...,...,...
13,14,1.000000,0.037037,rule,v1,2026-01-06T09:47:10.876447+00:00
5,3,1.000000,0.045455,rule,v1,2026-01-06T09:47:10.876447+00:00
19,20,1.000000,0.027027,rule,v1,2026-01-06T09:47:10.876447+00:00
48,49,1.000000,0.111111,rule,v1,2026-01-06T09:47:10.876447+00:00


In [69]:
review_entity_id = 26
id = df_re.loc[review_entity_id, 'review_id'].squeeze()
df_r.loc[id, 'content']

'정자동 정동마트에서 종종 바나나 세일을 하는데 대부분 거무죽죽하니 신선도 관리를 어찌 하는지 황당하다'

## 1) Review-level sentiment (multilingual transformer)

We use a multilingual RoBERTa sentiment model and run it over full reviews (not sentence-by-sentence).


In [63]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

sentiment_model_id = 'cardiffnlp/twitter-xlm-roberta-base-sentiment'
tokenizer = AutoTokenizer.from_pretrained(sentiment_model_id)
model = AutoModelForSequenceClassification.from_pretrained(
    sentiment_model_id,
    use_safetensors=True
)
sentiment_pipe = pipeline(
    'sentiment-analysis',
    model=model,
    tokenizer=tokenizer,
    truncation=True,
    max_length=256,
    batch_size=16
)

def analyze_sentiments(texts):
    # Keep order aligned with input
    results = sentiment_pipe(texts)
    return pd.DataFrame(results)

sentiment_df = analyze_sentiments(reviews['content'].tolist())

Device set to use cpu


In [64]:
sentiment_df = pd.concat([reviews[['id', 'content']], sentiment_df], axis=1)
sentiment_df.rename(columns={'id': 'review_id'}, inplace=True)
sentiment_df.head()

,review_id,content,label,score
0,1,BHC서현시범단지점. 근처 지점중 배달 최수 주문가가 가장 저렴해 치킨 주문. 쏘마...,positive,0.530003
1,2,처가집 양념통닭에서 마라치킨 주문. 그냥 달달한 양념 치킨에 마라향 살짝 추가한 정...,neutral,0.379968
2,3,배달앱에서 멕시카나 치킨 할인한다기에 주문하려했더니 미금점 영업을 하지 않는다. 이...,negative,0.635604
3,4,비비큐에서 맵소디 배달 주문. 양념 맛은 기억나지 않지만 소스 안쪽에서도 바삭한 식...,neutral,0.572736
4,5,노랑통닭은 순살도 가격이 더 비싸지 않아 좋다. 튀김 부스러기는 과자같이 바삭하니 ...,positive,0.694385


In [ ]:
sentiment_df['label'].value_counts(normalize=True).rename('share')

In [12]:
sentiment_df.sort_values(by=['label', 'score'], ascending=False)

,review_id,content,label,score
9,10,성남 현대시장 명성왕족발에서 족발 대자 포장. 싸고 맛 좋고 특히 온누리상품권을 쓸...,positive,0.814621
30,32,가격이 저렴해 구입. 가쓰오부시 덕분인지 국물이 두툼하니 생각보다 맛있었다.,positive,0.800935
27,29,새해 첫날 영업하길래 점심 방문. 실내가 깔끔해서 좋다. 상차림도 왠지 깔끔한 느낌...,positive,0.732063
4,5,노랑통닭은 순살도 가격이 더 비싸지 않아 좋다. 튀김 부스러기는 과자같이 바삭하니 ...,positive,0.694385
31,33,고기 두툼하고 잡내 없어 맛있다. 고기가 덜익은 건 아닌데 조리법보다 좀더 익혀도 ...,positive,0.681882
5,6,푸라닭에서 유명 쉐프와 협업했다는 깐풍치킨 주문. 소스가 볶은 느낌이 아니라 흥건하...,positive,0.643638
6,7,교촌치킨에서 간장 레드 반반 주문. 간장만 먹으면 심심하고 레드만 먹으면 자극적이면...,positive,0.634538
13,14,속초에서 시장 구경하고 돌아가는 길 닭강정 사는걸 깜빡했다. 많은 사람들이 만석닭강...,positive,0.628775
11,12,민생지원금으로 성남 할머니 족발에서 족발 포장. 잡내 없고 맛나다. 다음에 명성왕족...,positive,0.581646
0,1,BHC서현시범단지점. 근처 지점중 배달 최수 주문가가 가장 저렴해 치킨 주문. 쏘마...,positive,0.530003


## 2) Entity match + suggestions

We embed review text and entity labels, then compute cosine similarity.
- `selected_match_score`: mean similarity between the review and its selected entities.
- `suggested_entities`: top-k entities not already selected.


In [4]:
embedder_id = 'paraphrase-multilingual-MiniLM-L12-v2'
embedder = SentenceTransformer(embedder_id)

nodes = nodes.copy()
nodes['label'] = nodes.apply(lambda row: f"{row['name']} ({row['type']})", axis=1)

entity_labels = nodes['label'].tolist()
entity_embeddings = embedder.encode(entity_labels, convert_to_tensor=True, normalize_embeddings=True)

review_texts = reviews['content'].tolist()
review_embeddings = embedder.encode(review_texts, convert_to_tensor=True, normalize_embeddings=True)

review_entities_grouped = (
    review_entities.groupby('review_id')['node_id']
    .apply(list)
)

node_id_to_index = {node_id: i for i, node_id in enumerate(nodes['id'].tolist())}

def score_selected_entities(review_id, review_idx):
    node_ids = review_entities_grouped.get(review_id, [])
    if not node_ids:
        return np.nan
    indices = [node_id_to_index[nid] for nid in node_ids if nid in node_id_to_index]
    if not indices:
        return np.nan
    sims = util.cos_sim(review_embeddings[review_idx], entity_embeddings[indices]).cpu().numpy()
    return float(sims.mean())

def suggest_entities(review_id, review_idx, k=5):
    selected = set(review_entities_grouped.get(review_id, []))
    sims = util.cos_sim(review_embeddings[review_idx], entity_embeddings).cpu().numpy().ravel()
    ranked = sims.argsort()[::-1]
    suggestions = []
    for idx in ranked:
        node_id = nodes.iloc[idx]['id']
        if node_id in selected:
            continue
        suggestions.append({
            'node_id': int(node_id),
            'label': nodes.iloc[idx]['label'],
            'score': float(sims[idx])
        })
        if len(suggestions) >= k:
            break
    return suggestions

rows = []
for idx, row in reviews.iterrows():
    review_id = row['id']
    rows.append({
        'review_id': review_id,
        'selected_match_score': score_selected_entities(review_id, idx),
        'suggested_entities': suggest_entities(review_id, idx, k=5)
    })

suggestions_df = pd.DataFrame(rows)
suggestions_df.head()


,review_id,selected_match_score,suggested_entities
0,1,0.479157,"[{'node_id': 54, 'label': '닭강정 (menu)', 'score..."
1,2,0.252874,"[{'node_id': 54, 'label': '닭강정 (menu)', 'score..."
2,3,0.268003,"[{'node_id': 34, 'label': '정동마트 (vendor)', 'sc..."
3,4,0.388947,"[{'node_id': 47, 'label': '분당맛소 (vendor)', 'sc..."
4,5,0.638946,"[{'node_id': 54, 'label': '닭강정 (menu)', 'score..."


In [ ]:
df = suggestions_df.sort_values('selected_match_score')

In [15]:
review_id = 16
df.loc[review_id, 'suggested_entities']

[{'node_id': 29, 'label': '한사랑 정형외과의원 (vendor)', 'score': 0.46180397272109985},
 {'node_id': 30, 'label': '이주철마취과의원 (vendor)', 'score': 0.2528977394104004},
 {'node_id': 13, 'label': '자담치킨 (vendor)', 'score': 0.09404535591602325},
 {'node_id': 20, 'label': '굽네치킨 (vendor)', 'score': 0.07251551747322083},
 {'node_id': 43, 'label': '교촌치킨 (vendor)', 'score': 0.07195234298706055}]

In [16]:
reviews.set_index('id').loc[review_id, 'content']

'발뒤꿈치가 아파 정형외과를 찾는데 저번에 비싼 주사 맞은데를 피해서 다른 병원을 갔다. 갔더니 아픈 부위를 직접 보지도 않고 발모형에서 아픈 부위를 가리켜 보라고 한다. 뼈이상은 아닌것 같으니 엑스레이를 찍을 필요는 없다는 건 합리적이라 좋았지만. 물리치료 받으라는데 믿음이 안가 그냥 나왔따'

In [5]:
suggestions_df['selected_match_score'].describe()


count    25.000000
mean      0.357580
std       0.167580
min       0.068408
25%       0.231702
50%       0.363672
75%       0.457171
max       0.687745
Name: selected_match_score, dtype: float64

### Notes
- Increase `max_length` if you want to keep more of long reviews for sentiment.
- For larger datasets, embed in batches and persist embeddings to disk.
- Suggestions are purely semantic similarity; consider adding type or alias constraints if needed.
